# 🏦 Short-Term Interest Rate Contracts: The 3-Month SOFR Futures

<br>
<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Derivatives-Futures-Chapter-Four/blob/master/colab/Colab_short_term_contracts.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Derivatives-Futures-Chapter-Three/master?urlpath=lab/tree/notebooks/short_term_contracts.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Derivatives-Futures-Chapter-Four/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to the Chapter
  </a>
</div>

## 🌊 The Three Liquid Contracts
The Chicago Mercantile Exchange (CME) hosts three short-term interest rate (STIR) futures contracts that represent the bedrock of dollar funding:
- **3-Month SOFR (SR3):** Settles based on the compounded daily SOFR rate over a specific three-month quarterly reference period.
- **1-Month SOFR (SR1):** Settles based on the arithmetic average of daily SOFR rates (including weekends and holidays) for the named contract calendar month.
- **30-Day Federal Funds (ZQ):** Settles based on the arithmetic average of the daily Effective Federal Funds Rate (EFFR) for the named contract calendar month.

All three instruments track short-term dollar borrowing costs, but their structural differences—compounding versus arithmetic averaging—anchor them to distinctly different institutional applications.

Before constructing pricing models, this notebook establishes an empirical baseline by examining the **quantitative liquidity profiles** of each contract across near-term and multi-year tenors. In STIR markets, liquidity is the purest signal of institutional adoption; measuring where volume persists along the forward curve dictates which instrument can reliably support multi-year hedging, synthetic bond replication, and yield curve bootstrapping.

## 🛠️ The Necessary Data

The Notebook will use both FRED and MASSIVE data.  The first order of business is importing the financial_quant package and the libraries Pandas and NumPy.

## 📦 Getting Functions from financial_quant package

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;"><span style="font-size: 1.4em;">⚡🔌</span> Professional Packaging: How GitHub Installations Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library updater script.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch & Build:</b> The <code>urllib</code> script downloads and executes <code>fq_updater.py</code>, which verifies local dependency versions (like locking Pandas to 2.2.3) and installs or updates the <code>financial_quant</code> package in the background.</li>
<li><b>Import:</b> <code>fq = import_financial_quant()</code> loads the verified package into your notebook's memory and assigns it the quick alias <code>fq</code>.</li>
<li><b>Routing:</b> Behind the scenes, a special file called <code>__init__.py</code> acts as the package's "front door." It automatically gathers complex tools from deeply nested folders (like our fixed-income models and chart visualizers) and serves them up directly to the surface.</li>
<li><b>Execute:</b> You don't have to worry about where the files live. You just type <code>fq.create_arb_vix()</code> or <code>fq.plot_contract_arb_band()</code>, and Python immediately knows where to route the request.</li>
</ol>

<p><b>Why do this?</b><br>
The most important reason is to ensure these notebooks are environment-agnostic—whether you are using Jupyter Lab, Google Colab, or classic Jupyter Notebooks. Using the <code>financial_quant</code> package allows us to align all software versions and interpolation rules across machines. The package installs and manages the libraries; you only have to import them.</p>
</details>

In [ ]:
# Use Python's built-in urllib to fetch the remote setup script
import urllib.request

# Download, decode, and execute the updater script directly from GitHub
updater_url = "https://raw.githubusercontent.com/PatrickJHess/quant_repo/master/fq_updater.py"
exec(urllib.request.urlopen(updater_url).read().decode('utf-8'))

# Run the installer/updater function and alias the package as 'fq'
fq = import_financial_quant()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import altair as alt
from IPython.display import clear_output

## 🚰 Fetching the Data: SOFR Futures and SOFR Index

We will use three custom data pipelines—`FredReader`,`MASSIVEReader`, and `static_futures_data` to pull the necessary data for this notebook. All three are included in the `financial_quant` package that we imported earlier as `fq`.

To divide and conquer the workload, we will instantiate two specific data objects:

* `fred_data`: Powered by `FredReader`, this fetches the SOFR and VIX Indexes as well as the spot price of the S&P 500.
*  `massive_data`: Powered by `MASSIVEReader`, this serves as our primary engine for downloading the daily futures contract data.
* `static_futures_data`: This module supplies the S&P 500 Daily Rates of Return needed for our baseline calculations.

In [ ]:
# Instantiate the data pipelines from the financial_quant package
fred_data = fq.FredReader()
massive_data = fq.MASSIVEReader(calls_per_minute=5)
static_futures_data=fq.static_futures_data

## 📊 Gauging the Liquidity Profile of the Contracts

We examine the liquidity over the past sixty days for the near-term three-month SOFR contract, alongside the corresponding quarterly contracts expiring in one and two years. The same expiration months are used for the Fed funds and one-month SOFR contracts. We also look at the liquidity of these contracts one and two years out. Because the one-month SOFR contract only lists for twenty four months forward, the two-year-ahead contract is often unavailable. When this occurs, the missing contract is simply represented as empty data in our summary tables.

### ⚙️ Automating Contract Retrieval and Validation
The get_contracts function automates the retrieval of historical futures data for a specific contract code across three time horizons: the current year (near contract), one year out, and two years out.

First, it establishes a rolling lookback window—defaulting to 60 business days—up to the current date. It then dynamically constructs the correct futures ticker symbols for each timeframe by appending the current quarter's letter code (H, M, U, or Z) and the last digit of the target year.

Crucially, the function is built to handle missing data gracefully while maintaining visibility. Because longer-dated contracts (like the two-year forward three-month SOFR) are sometimes unavailable, the data fetch is wrapped in a `try-except block`. If a contract is missing or the API returns an error, the function prints a warning to the console detailing the specific exception. It then automatically substitutes an empty DataFrame pre-populated with a float-type volume column. This dual safeguard ensures users are alerted to missing data, while the function always outputs exactly three structured DataFrames, allowing downstream concatenation and statistical summaries to execute flawlessly without `KeyError` or `IndexError` exceptions.

To avoid overwhelming the user with terminal output, the get_contracts function includes an option to clear execution logs after the data is accessed. The default value of clear_logs (False) leaves all notifications intact, while setting it to True suppresses them.

**Anchoring to Valid Market Days**

Before executing any API calls, `get_contracts` relies on the `set_settlement` utility function to validate and sanitize the requested anchor date. Whether a user inputs a custom date or the function defaults to the current day, `set_settlement` verifies that the target is an active trading session. If the date falls on a weekend, a Federal Reserve holiday, or a CME-specific closure like Good Friday, the function dynamically rolls it backward to the nearest valid settlement day. This critical sanitization step ensures the lookback window is always anchored accurately and completely protects the data pipeline from malformed inputs and non-market days.

---


> 💡 **The `clear_output()` function**
>
>`clear_output()` is a function from the IPython.display module used to remove the current output of a Jupyter Notebook cell or output widget during runtime.
> `clear_output(wait=False)`(the default) clears the output immediately.  Setting `wait=True` delays clearing until new output is ready to be displayed.

In [ ]:
def set_settlement(date):
    """
    How it works:
      1. Type Enforcement: Attempts to cast the input to a pd.Timestamp, raising
        a custom ValueError if the input is malformed or unparseable.
      2. Holiday Detection: Passes the date to fq.adjust_bond_pay_dates(). Because
        that function pushes closures forward, any returned date strictly greater
        than the input indicates the input was a market closure.
      3. Iterative Rollback: Subtracts one day and re-tests until the adjusted date
        matches the target date.
    """
    # Validate the date
    try:
        date = pd.Timestamp(date)
    except (ValueError, TypeError):
        raise ValueError(f"Garbage input detected: '{date}'. Expected a valid date string, datetime, or Timestamp.")

    # fq.adjust_bond_pay_dates uses a "Following Business Day" convention.
    # If the adjusted date is pushed forward (> date), the current date is a closure.
    # We subtract one day and re-test until we land on a valid preceding market day.
    while pd.Timestamp(fq.adjust_bond_pay_dates(date)[0]) > date:
        date += pd.Timedelta(days=-1)

    return date

In [ ]:
def get_contracts(code, date=None, number_days=60,
                  clear_logs=False,
                  wait_to_clear=False):
    """
    Fetches the near, 1-year forward, and 2-year forward futures contracts.
    """
    # 1. Handle default and type casting
    is_custom_date = date is not None
    raw_date = pd.Timestamp.now() if date is None else pd.Timestamp(date)

    # 2. Check for valid settlement date
    anchor_date = set_settlement(raw_date)

    # Alert the user if the date they entered was a holiday/weekend and had to be moved
    if is_custom_date and anchor_date != raw_date:
        print(f"Warning: {raw_date.strftime('%Y-%m-%d')} is a holiday or weekend. "
              f"Adjusting to the nearest valid settlement date: {anchor_date.strftime('%Y-%m-%d')}")

    # 3. Calculate lookback window
    start = anchor_date - (number_days * pd.offsets.BDay())

    # Ensure the lookback window starts on a business day
    while pd.Timestamp(fq.adjust_bond_pay_dates(start)[0]) > start:
        start += pd.Timedelta(days=-1)

    start_date = start.strftime('%Y-%m-%d')
    end_date = anchor_date.strftime('%Y-%m-%d')

    # 4. Map the anchor date's calendar quarter
    quarter_codes = ['H', 'M', 'U', 'Z']
    current_quarter = quarter_codes[anchor_date.quarter - 1]

    results = []

    for add_date in [0, 1, 2]:
        target_year = str(anchor_date.year + add_date)[-1]
        symbol = code + current_quarter + target_year

        try:
            df = massive_data.get_futures_data(symbol, start_date, end_date)

            if not isinstance(df, pd.DataFrame):
                df = pd.DataFrame(columns=['volume'], dtype=float)
            elif 'volume' not in df.columns:
                df['volume'] = pd.Series(dtype=float)

        except Exception as e:
            print(f"Warning: Failed to retrieve {symbol}. Reason: {e}")
            df = pd.DataFrame(columns=['volume'], dtype=float)

        results.append(df)

    if clear_logs:
        clear_output(wait=wait_to_clear)

    near, plus_one, plus_two = results

    return near, plus_one, plus_two

In [ ]:
sofr3_near, sofr3_plus_one, sofr3_plus_two = get_contracts('SR3',clear_logs=True)
sofr1_near, sofr1_plus_one, sofr1_plus_two = get_contracts('SR1',clear_logs=True)
fed_near, fed_plus_one, fed_plus_two = get_contracts('ZQ',clear_logs=True)

### 📊 Generating Hierarchical Volume Summaries
The `create_contract_summary` function transforms the raw time-series data retrieved by `get_contracts` into a clean, statistical overview of trading volume across the curve.

First, the function isolates the `volume` column from the near, one-year, and two-year DataFrames and concatenates them horizontally. Because the upstream  function `get_contracts` guarantees that a `volume` column always exists—even for missing or unlisted contracts—this concatenation is entirely safe from `KeyError` exceptions. By chaining pandas' built-in `.describe()` and `.round()` methods, the function instantly generates standard descriptive statistics (such as count, mean, standard deviation, and quartiles) for all three timeframes simultaneously. Empty dataframes are seamlessly handled, yielding `NaN` values rather than breaking the script.

To ensure the final output is intuitive and presentation-ready, the function applies a pandas `MultiIndex` to the columns. This creates a two-tiered hierarchical header: the top level identifies the underlying asset (passed via the `contract_name` parameter), while the secondary level specifies the respective timeframe ("Near", "1 Year", "2 Years"). This structural design is particularly advantageous for reporting, as it allows summaries from multiple different asset classes to be merged side-by-side into a single, highly readable master table.


---
>💡 **The pd.MultiIndex.from_tuples() method**

>A standard Pandas DataFrame has a single, flat row of column headers. The MultiIndex.from_tuples() method allows you to build hierarchical (stacked) headers by passing a list of tuple pairs.
>* **The first item in the tuple** creates a grouped, top-level category (e.g., "1-Month SOFR").
> * **The second item** creates the specific sub-columns beneath it (e.g., "Near", "1 Year").
>
>Pandas automatically merges the matching top-level names, turning plain data into a professional, presentation-ready financial table.


In [ ]:
def create_contract_summary(near, plus_one, plus_two, contract_name):
    """
    Generates descriptive statistics for trading volume across three time horizons.
    Applies a two-tiered column index (Asset Name -> Timeframe) for standardized reporting.
    """
    # Horizontally align the volume series; safe to execute even if upstream data is empty
    summary = pd.concat([
        near['volume'],
        plus_one['volume'],
        plus_two['volume']
    ], axis=1).describe().round()

    # Build a hierarchical header to allow multiple asset summaries to merge seamlessly
    summary.columns = pd.MultiIndex.from_tuples([
        (contract_name, 'Near'),
        (contract_name, '1 Year'),
        (contract_name, '2 Years')
    ])

    return summary

### 🎨 Assembling and Styling the Master Summary Table
With the underlying data retrieved, we leverage our create_contract_summary helper function to generate statistical profiles for the 3-Month SOFR, 1-Month SOFR, and Fed Funds contracts. By concatenating these individual summaries horizontally along the column axis, we create a unified master dataset. Thanks to the hierarchical `MultiIndex` established in the previous step, the columns align perfectly under their respective asset headers without naming collisions.

To transform this dense numeric output into a presentation-ready report, we apply the pandas `.style` method, chaining together several visual enhancements:

* **Contextual Titling:** We inject a custom, bolded caption directly into the table's HTML properties to establish clear context for the reader.

* **Visual Segmentation:** Using `pd.IndexSlice`, we dynamically target the top level of our `MultiIndex` to color-code the data by asset class (cyan for 3-Month SOFR, coral for 1-Month SOFR, and yellow for Fed Funds). This creates distinct vertical zones, allowing the eye to easily separate the contracts.

* **Numeric Clarity:** We apply a global formatting string (`"{:,.0f}"`) to insert thousand separators and strip unnecessary decimal places, significantly improving the readability of large volume figures.

---
>💡 Targeting MultiIndex Columns with `pd.IndexSlice`

>Selecting data in a standard DataFrame is simple (e.g., `df['Column']`). But when your table has hierarchical, stacked headers, standard bracket notation falls short.

>`pd.IndexSlice` (often assigned to the short variable `idx`) acts as a specialized slicer for these multi-tiered layouts. In our styling code `subset=idx[:, idx['Fed Funds', :]]`:

>The first : tells Pandas to apply the style to all rows.

>The second part, `idx['Fed Funds', :]`, tells Pandas to search the columns, grab the top-level 'Fed Funds' category, and select all sub-columns (Near, 1 Year, 2 Years) nested beneath it.


In [ ]:
# Create a master summary of all three contracts
master_summary=pd.concat([
    create_contract_summary(sofr3_near, sofr3_plus_one, sofr3_plus_two, 'SOFR Three Months'),
    create_contract_summary(sofr1_near, sofr1_plus_one, sofr1_plus_two, 'SOFR One Month'),
    create_contract_summary(fed_near, fed_plus_one, fed_plus_two, 'Fed Funds')
], axis=1)

# Display the final master table
idx = pd.IndexSlice

# Chain the styling properties together
styled_summary = (master_summary.style
    .set_caption("Liquidity Summary for Interest Rate Futures")
    .set_table_styles([{
        'selector': 'caption',
        'props': [('caption-side', 'top'),
                  ('font-size', '16px'),
                  ('font-weight', 'bold'),
                  ('color', 'black')]
    }])
    .set_properties(subset=idx[:, idx['SOFR Three Months', :]], **{'background-color': 'cyan'})
    .set_properties(subset=idx[:, idx['Fed Funds', :]], **{'background-color': 'yellow'})
    .set_properties(subset=idx[:, idx['SOFR One Month', :]], **{'background-color': 'coral'})
    .format("{:,.0f}")
)

# Display the styled table
styled_summary

### 📉 Summarizing the Liquidity Tenor of the Contracts

The empirical data highlights a stark contrast in institutional utility:
* **3-Month SOFR (SR3)** sustains deep, institutional liquidity out to multi-year horizons—maintaining active trading across all observation windows and averaging tens of thousands of contracts daily even two years forward.
* **1-Month SOFR (SR1) and Fed Funds (ZQ)** see forward liquidity evaporate almost immediately. Beyond a few months, 1-Month SOFR trading functionally ceases, and forward Fed Funds activity collapses past the 6-month horizon.

This drop-off defines their respective roles: while 1-Month SOFR and Fed Funds serve as tactical tools for short-term policy positioning and immediate reserve management, **3-Month SOFR is the undisputed benchmark for multi-year term financing and curve construction**.

Next, we explore the macroeconomic and regulatory architecture driving this liquidity—specifically how institutions deploy 3-Month SOFR futures strips to hedge bespoke swaps, maximize capital efficiency, and engineer synthetic fixed-rate curves.

## 🚰 Expanding the Plumbing: LIBOR, Swaps, and Capital Efficiency

**The LIBOR Legacy and the Need for Compounding**

The sustained liquidity we observe in the three-month SOFR contract is deeply tied to the structural overhaul of global financial plumbing—specifically, the transition away from LIBOR and the regulatory shift in how derivatives are collateralized.

Historically, the financial system relied on LIBOR and its forward-looking term rates (e.g., one-month, three-month). SOFR, however, is strictly an overnight rate based on repo transactions of Treasury securities. Because it is the safest rate in financial markets, it serves as the natural base rate for financing a trade or debt. Lenders and borrowers use SOFR as a baseline and then adjust for credit risk and term. To accommodate institutional demands, the base rate for these loans is set as the daily compounded SOFR over a given quarter. The three-month SOFR futures contract effectively packages this compounded rate, making it an indispensable tool for locking in term financing costs.

<details>
<summary><b style="font-size: 1.1em; color: #1976d2; cursor: pointer;">💡 If You Owned A Treasury Bond, How Could You Fund A Loan Without Selling The Bond?</b></summary>

> The repo market provides the solution. When you repo a bond, three things happen:
>
> 1. Using the bond as security, you borrow approximately 100% of the value of the bond for one day.
> 2. You agree to repurchase the bond the next day for the amount borrowed plus the SOFR rate times the amount borrowed.
> 3. You repeat the repo as long as the loan is outstanding.
>
> What are you paying to fund the loan? SOFR.
>
> What quarterly rate should you demand on your loan? The compounded SOFR rate, plus a premium for your efforts and the riskiness of the loan.

</details>

**Bespoke OTC Swaps and Institutional Block Sizes**

Beyond simple borrowing, the enormous Over-The-Counter (OTC) interest rate swap market relies on this three-month tenor. It is worth noting that, much like total return swaps on the S&P 500, these OTC interest rate swaps are bespoke agreements. They are custom-tailored to specific dates and exact cash flows, meaning they are held on the books rather than being actively bought and sold on an exchange like standardized futures. Furthermore, both types of swaps involve large minimum block sizes, effectively restricting them to institutional scale.

**The Clearing Advantage and Cross-Hedging at the CME**

The 2008 financial crisis triggered new regulations that fundamentally changed the economics of OTC swaps. Regulators heavily increased the capital requirements for uncleared swaps, making them far less profitable to hold. To avoid this punitive cost of capital, banks now migrate their swaps to central clearinghouses like the CME. It is critical to distinguish clearing from trading here. Although the clearinghouse handles the daily settlement of margin as the swap's value fluctuates, it does not create an active secondary market. The swaps themselves remain customized, untraded, and held on the books; clearing merely centralizes the counterparty risk and greatly diminishes the capital cost.

This is where the deep liquidity of the three-month SOFR futures contract becomes crucial. If an institution holds cleared OTC swaps and simultaneously executes highly liquid, standardized three-month SOFR futures as offset trades at the CME, the clearinghouse recognizes the balanced risk. Through "cross-margining" or "cross-hedging," the CME allows participants to post significantly less margin. This capital efficiency creates a powerful feedback loop: institutions flock to the three-month SOFR futures to efficiently hedge their bespoke swap books, directly resulting in the contract's deep, long-term liquidity.

## ⚖️ Hedging Compounded SOFR Rates

Although we have referred to the three-month SOFR contracts in previous notebooks, it is worth reviewing how a single contract hedges borrowing costs and lending returns linked to SOFR.

**The Pricing Convention**

Three-month SOFR futures trade on an IMM (International Monetary Market) index convention:

$$\text{Price} = 100 - R$$

where $R$ is the annualized, implied daily compounded SOFR rate for the contract's reference quarter. For example, if a futures contract settles at **96.00**, the market's fair expectation for the compounded value of SOFR over that quarter is **4.00%**.

**Hedging the Compounded Value of SOFR**

Because the contract price moves inversely to interest rates, hedging requires matching your cash-market rate delta with an opposite futures delta:
* **The Borrower (Short Hedge):** An entity paying floating daily SOFR faces the risk of rising rates. To offset higher cash interest expenses, the borrower sells futures (**short position**). As rates climb, the futures price drops, generating a gain that covers the higher funding cost.
* **The Lender (Long Hedge):** An entity earning floating daily SOFR (for example, by lending cash against Treasury collateral in a reverse repo) faces the risk of declining rates. To offset falling interest income, the lender buys futures (**long position**). As rates fall, the futures price rises, generating a gain that replaces lost cash yield.

The table below details these mechanical offsets across both rate environments:

| Regime | Mechanics | The Borrower | The Lender |
| :--- | :--- | :--- | :--- |
| **General Setup** | Core Exposure | Pays floating daily SOFR | Earns floating daily SOFR |
| **General Setup** | Required Hedge | Short position | Long position |
| | | | |
| <span style="color: #B22222;">**Rising Rates**</span> | <span style="color: #B22222;">Floating Cash Flow</span> | <span style="color: #B22222;">Interest expense increases</span> | <span style="color: #B22222;">Interest income increases</span> |
| <span style="color: #B22222;">**Rising Rates**</span> | <span style="color: #B22222;">Futures Offset</span> | <span style="color: #B22222;">Short position gains value</span> | <span style="color: #B22222;">Long position loses value</span> |
| <span style="color: #B22222;">**Rising Rates**</span> | <span style="color: #B22222;">Net Hedged Result</span> | <span style="color: #B22222;">Futures profit covers higher interest expense</span> | <span style="color: #B22222;">Higher cash interest income covers futures loss</span> |
| | | | |
| <span style="color: #00509E;">**Falling Rates**</span> | <span style="color: #00509E;">Floating Cash Flow</span> | <span style="color: #00509E;">Interest expense decreases</span> | <span style="color: #00509E;">Interest income decreases</span> |
| <span style="color: #00509E;">**Falling Rates**</span> | <span style="color: #00509E;">Futures Offset</span> | <span style="color: #00509E;">Short position loses value</span> | <span style="color: #00509E;">Long position gains value</span> |
| <span style="color: #00509E;">**Falling Rates**</span> | <span style="color: #00509E;">Net Hedged Result</span> | <span style="color: #00509E;">Lower cash interest expense covers futures loss</span> | <span style="color: #00509E;">Futures profit covers lower interest income</span> |
| | | | |
| **Ultimate Outcome** | **Result** | **Effective borrowing cost locked in** | **Effective lending return locked in** |

By executing this hedge, market participants eliminate quarterly rate volatility. Regardless of where daily SOFR prints over the accrual window, the effective borrowing cost or lending yield for that specific quarter is locked in at the rate implied by the futures price on the trade date.

## 🧱 Stacking Contracts: The Futures Strip and Known Coupons

A single three-month SOFR contract hedges a specific calendar quarter, but that doesn't match institutional needs—namely borrowing and lending for multiple quarters or years. Hedging has to match the floating-rate exposure, requiring a sequence of consecutive quarterly contracts, known as a futures "strip."

**Locking in the Forward Curve**

Stacking consecutive contracts systematically hedges each upcoming quarter during the term of the borrowing and lending. Because futures prices reflect the fair bet of compounded SOFR, a strip locks in the values for each quarter during the term.

**Engineering Synthetic Bonds & Interest Rate Swaps**

This sequence of locked-in forward rates fundamentally changes the nature of the underlying financial exposure:
- **The Transformation:** A floating-rate loan or investment with unknown future cash flows is transformed into an instrument with entirely predictable cash flows.
- **Known, But Variable:** Keep in mind that the fair bet of compounded SOFR varies from quarter to quarter. The coupon of the bond is known, but it's not the same in every quarter.
- **The Ultimate Objective—Pricing the Swap:** Exchanging a stream of variable cash flows for a single, equivalent fixed rate over a multi-year term is the exact definition of an Interest Rate Swap (IRS). Our ultimate goal in stacking these contracts is to calculate the **par swap rate**—the single fixed yield that perfectly balances out this string of variable forward rates.

Once these sequential quarterly cash flows are locked in and mapped, the synthetic asset can be valued, and these fixed swap rates can be extracted using standard fixed-income mathematics.

**Mapping the IMM Timeline**

To extract these swap rates, we must first map the exact timeline of our futures strip. Since 3-Month SOFR contracts are anchored to standard quarterly IMM dates (the third Wednesday of March, June, September, and December), our first programming step is to generate these precise contract periods and calculate the exact day counts required for our fixed-income math.


In [ ]:
# Build the quarterly IMM schedule: Sep 2026 (U26) to Jun 2030 (M30)
# 'WOM-3WED' generates the 3rd Wednesday of the specified months
anchor_date = pd.Timestamp('2026-10-27')
final_date = pd.Timestamp('2031-12-30')

# Push the search start date back to catch the IMM date prior to our anchor
start_search_date = anchor_date - pd.DateOffset(months=3)

# 'WOM-3WED' generates the 3rd Wednesday of the specified months
all_3wed = pd.date_range(start=start_search_date, end=final_date, freq='WOM-3WED')

# Filter to keep ONLY the quarterly IMM months (March, June, September, December)
imm_dates = np.array([wed for wed in all_3wed if wed.month in [3, 6, 9, 12]])

# True reference starts and ends (Needed for N and querying Index_0)
start_dates = imm_dates[:-1]
end_dates = imm_dates[1:]

# Calculate N (total days in reference period)
N_days = np.array([(end - start).days for start, end in zip(start_dates, end_dates)])

# Calculate k (elapsed days inside the reference period).
# This correctly assigns 41 days to the front contract and 0 to the rest.
k_days = np.array([
    (anchor_date - start).days if anchor_date > start else 0
    for start in start_dates
])

# Create a clean index for our dataframe representing the contract delivery months
contract_months=['Sep. 2026','Dec. 2026',
       'Mar. 2027','Jun. 2027','Sep. 2027','Dec. 2027',
       'Mar. 2028','Jun. 2028','Sep. 2028','Dec. 2028',
       'Mar. 2029','Jun. 2029','Sep. 2029','Dec. 2029',
       'Mar. 2030','Jun. 2030','Sep. 2030','Dec 2030',
       'Mar. 2031','Jun. 2031','Sep. 2031']

# Initialize the master dataframe holding all term structure data
df_contracts=pd.DataFrame({'Start': start_dates, 'End': end_dates,
              'Reference Period': N_days,
              'Realized Days': k_days},
            index=contract_months)

## 🔄 A Recollection: SOFR Futures to Forward Rates

As a reminder, in Chapter Three we used the following formula to convert futures prices into forward rates:

$$R^{forward}_{k} = \left( \left(1 + R^{futures}_{k} \times \frac{N}{360}\right) \times \large\frac{\text{SOFR Index}_{0}}{\text{SOFR Index}_{k}} - 1 \right) \times \large\frac{360}{N-k}$$

The application of this formula depends entirely on whether we are pricing the contract **before** or **during** its reference period:

*   **Before the Reference Period ($k = 0$):** For all deferred contracts (every contract except September 2026 in our example), the reference period has not yet started. In these cases, the equation simplifies gracefully, and the implied forward rate is exactly $100 - \text{Future Price}$, or $R^{futures}_{0}$.

*   **During the Reference Period ($k > 0$):** For the front September 2026 contract, our anchor date (October 27, 2026) falls *inside* the accrual period. Because days have already elapsed since the reference period began on September 16, 2026, $k > 0$. The realized SOFR index for those elapsed days must be factored into the equation to isolate the remaining forward expectation.

The code cell below adds the `SOFR Ratio` to the `df_contracts` Dataframe.  

In [ ]:
# ==============================================================================
#  CALCULATE THE SOFR INDEX RATIO FOR REALIZED DAYS
# ==============================================================================
# Fetch the daily SOFR Index values from FRED for our timeline
sofr_index = fred_data.get_series('SOFRINDEX', start_date='2026-09-16', end_date='2031-12-30')

# Extract the index value on the front contract's start date (Index_0)
sofr_index_9_16 = sofr_index.loc['2026-09-16']

# Extract the index value on our valuation/anchor date (Index_k)
sofr_index_10_27 = sofr_index.loc['2026-10-27']

# Calculate the SOFR ratio: (Index_0 / Index_k)
# Use np.where to apply this ratio ONLY to the front contract (where start date is
# in the past). All deferred contracts get a clean ratio of 1.0.
sofr_index_ratios = np.where(start_dates <= anchor_date, sofr_index_9_16 / sofr_index_10_27, 1.0)

# Add the ratio column to our master dataframe
df_contracts['SOFR Ratio'] = sofr_index_ratios
df_contracts.head()

## 📊 Fetching Futures Prices: Constructing Ticker Symbols
With the time components ($N$, $k$) and the SOFR Ratio established, the final missing variable for our forward rate calculation is the futures price itself ($R^{futures}_{k}$).To query the historical database for the October 27, 2026 prices, we first need to construct the standard CME ticker symbols for the 3-Month SOFR futures strip. The convention uses the root SR3 followed by a standardized delivery month code and the last digit of the year. For the quarterly IMM cycle, the relevant month codes are:H: MarchM: JuneU: SeptemberZ: December(For example, the September 2026 contract becomes SR3U6).The code cell below maps our contract start dates to their respective month codes, generates the ticker symbols, fetches the settlement prices for our anchor date, and appends them to the dataframe.

In [ ]:
# ==============================================================================
# 3. GENERATE TICKER SYMBOLS AND FETCH SETTLEMENT PRICES
# ==============================================================================
# Map standard CME month codes to integer months (1=Jan, 2=Feb, etc.)
month_codes = ['F','G','H','J','K','M','N','Q','U','V','X','Z']

# Extract the integer month from the contract start dates
months = df_contracts['Start'].dt.month

# Convert the integer months to their corresponding CME letter codes
months = months.apply(lambda x: month_codes[x-1])

# Extract the last digit of the contract year (e.g., 2026 -> 6)
years = df_contracts['Start'].dt.year % 10

# Concatenate 'SR3' + Month Code + Year Digit to build the standard CME tickers
contract_symbols = 'SR3' + months + years.astype(str)

# Initialize a list to hold the fetched prices
settlement_prices = []

# Loop through each ticker symbol to fetch its settlement price on the anchor date
for symbol in contract_symbols.values:
    # Query the historical database for the specific symbol and date
    data = massive_data.get_futures_data(
        symbol,
        anchor_date.strftime('%Y-%m-%d'),
        anchor_date.strftime('%Y-%m-%d')
    )
    # Extract the scalar price value and append to our list
    settlement_prices.append(data['settlement_price'].values[0])

# Clear any messy API printouts from the loop to keep the notebook clean
clear_output(wait=False)

# Add the fetched prices to our master dataframe
df_contracts['Settlement Price'] = settlement_prices
df_contracts.head()

## 🧮 Calculating the Synthetic Coupons
With the futures settlement prices and our SOFR Ratio in place, we can now calculate the exact cash flow return—the synthetic coupon—for each quarter in the strip.

The beauty of the SOFR Ratio calculation from the previous step is that it allows us to use a single, cleanly vectorized operation across the entire dataframe:

* For the **deferred contracts**, the ratio is exactly 1.0, meaning the forward rate evaluates simply to 100 - Settlement Price.

* For the **front contract**, the fractional ratio (e.g., 0.995596) automatically scales down the implied rate to mathematically strip out the interest already realized over the elapsed days.

Once the annualized forward rate is calculated, we de-annualize it based on the exact number of remaining days in the period to find the actual percentage payout for that specific timeframe.

In [ ]:
# Calculate the adjusted annualized forward rate.
# For deferred contracts (Ratio = 1), this is simply 100 - Price.
# For the front contract, the ratio strips out the realized past interest.
forward_rate = (100 - df_contracts['Settlement Price']) * df_contracts['SOFR Ratio']

# Calculate the exact number of days remaining in the contract's reference period
remaining_days = df_contracts['Reference Period'] - df_contracts['Realized Days']

# De-annualize the forward rate using the Actual/360 money market convention
# to find the raw coupon return for this specific time block
synthetic_coupon = forward_rate * remaining_days / 360

# Append the final synthetic coupon to the dataframe
df_contracts['Synthetic Coupon'] = synthetic_coupon

# Display the top rows to verify the front contract vs deferred contracts
df_contracts.head()

## 💵 Constructing the Synthetic Bond Portfolio
By calculating the synthetic coupons, we have effectively transformed our futures strip into a series of predictable, fixed cash flows. We can now use these known coupons to engineer a portfolio of 22 synthetic bonds—each tied to a specific maturity date along our curve, with a standard par value of 100.

To anchor the very front of our curve, the first bond represents the immediate overnight cash rate, while the remaining 21 bonds are built from the quarterly futures strip:

* **Bond 1 (The Cash Rate - Matures Oct 28, 2026):** This bond represents the overnight realization of SOFR. It spans just one day, returning the 100 par value plus one day of accrued overnight interest. On our October 27 anchor date, the SOFR print was 3.95%. De-annualized for a single day using the Actual/360 convention, the math is:
  * $100 +\large\frac{3.95}{360}$
  * Final Payoff : 100.01097

* **Bond 2 (Matures Dec 16, 2026):** This bond is built from the remaining days of our front futures contract. On its maturity date, it pays back the 100 par value plus the locked-in synthetic coupon for that specific period (0.544121).

  * Final Payoff: 100.544121

* **Bond 3 (Matures Mar 17, 2027):** This bond spans two quarterly periods. On the first IMM date (Dec 16), it pays out the first synthetic coupon of 0.544121. On its maturity date in March, it pays back the 100 par value plus the second synthetic coupon (1.0619361).

  * First Cash Flow: 0.544121

  * Final Payoff: 101.0619361

* **Bonds 4 through 22:** This cascading pattern continues all the way down the curve. A synthetic bond maturing in 2030 will pay the sequence of known synthetic coupons on every single quarterly IMM date until its specific maturity, where it finally returns par plus that final quarter's coupon.

**The Cash Flow Matrix**

Mapping these payoffs creates a lower-triangular matrix of cash flows. Every row represents a unique bond, and every column represents a payment date. By including the overnight SOFR bond, we ensure our matrix seamlessly connects the present-day cash market directly to the futures curve.

This matrix is the fundamental building block of fixed-income engineering. Once you have a matrix of risk-free, known cash flows, you can use standard linear algebra to bootstrap a zero-coupon discount curve, price interest rate swaps, or value any arbitrary stream of future liabilities.

Before we calculate the payoff matrix, the overnight SOFR rate is added to the DataFrame.

In [ ]:
# ==============================================================================
# ADD OVERNIGHT SOFR TO ANCHOR THE CURVE
# ==============================================================================

# Define the columns for our future payoff matrix using the exact payout dates:
# The first date is tomorrow (for the overnight SOFR), followed by all IMM end dates.
columns = [pd.Timestamp('2026-10-28')] + df_contracts['End'].tolist()

# Initialize the empty payoff matrix with these dates as columns
df_payoffs = pd.DataFrame(columns=columns)

# Create a single-row dataframe for the overnight SOFR rate.
# We reverse-engineer the "Settlement Price" (100 - 3.95) so it matches the
# structure of the futures, and plug in the 1-day synthetic coupon calculated above.
df_sofr = pd.DataFrame({
    'Start': anchor_date,
    'End': pd.Timestamp('2026-10-28'),
    'Reference Period': 1,
    'Realized Days': 0,
    'SOFR Ratio': 1,
    'Settlement Price': 100 - 3.95,
    'Synthetic Coupon': 0.01097
}, index=['SOFR'])

# Concatenate the overnight rate to the top of our master contracts dataframe
df_contracts = pd.concat([df_sofr, df_contracts])

# Display the top rows to verify SOFR sits cleanly above the Sep 2026 contract
df_contracts.head()

## 🚰 The Payoff Matrix: The Essential Plumbing

To reach our ultimate goal of calculating single, fixed **par swap rates** for various multi-year tenors, we first need to extract a continuous curve of discount factors from our known cash flows.

If we already knew all these present value (PV) discount factors, calculating the theoretical prices ($P$) of our synthetic bonds would be a simple matrix multiplication:

<br>

$$P = \text{Payoff Matrix} \times \text{PV Factors}$$

<br>

Those PV factors, however, are unknown. What we actually need to estimate are the underlying spot rates of interest for the specific dates corresponding to the columns of our payoff matrix. The Nelson-Siegel model is the key that unlocks this door. The restricted version of the Nelson-Siegel model defines the spot rate $r(t)$ as:

<br>

$$r(t) = \beta_0 + (r(0) - \beta_0)\left( \frac{1 - e^{-t/\tau}}{t/\tau} \right) + \beta_2 \left( \frac{1 - e^{-t/\tau}}{t/\tau} - e^{-t/\tau} \right)$$

<br>

🔄 As a reminder, because the $\tau$ parameter enters this equation non-linearly to dictate the curve's shape, we cannot solve this with a simple linear regression. Instead, our estimation process iterates on this model. We make continuous guesses for the values of $\beta_0$, $\beta_2$, and $\tau$, using them to generate spot rates (and thus our PV factors). We push those factors through the payoff matrix to get our forecasted prices, repeating this loop until the sum of squared differences between the forecasted prices and the actual market prices is minimized.

> 💡 **What, conceptually, are we doing here?**
>
> In the first Chapter of *The Term Structure of Interest Rates* volume, we showed that the no-arbitrage condition implies that bond prices must equal their cash flows discounted at the corresponding spot rates. That is the exact principle driving this entire exercise! By forcing the theoretical prices of our synthetic bonds to equal exactly 100 (par), we can reverse-engineer the exact discount curve required to price an interest rate swap.

<details>
<summary><b>🔍 Deep Dive: Splines vs. Nelson-Siegel & The Mathematical Limits</b></summary>
<br>

### Why Not Just Interpolate?
When faced with gaps in a yield curve (e.g., having data for a 5-year and 7-year bond, but missing the 6-year), a common approach is to apply a mathematical interpolation like a cubic spline. While splines create a smooth, continuous curve, they are purely mathematical constructions—meaning they ignore the underlying economic structure of the data.

Because splines require calculating new polynomial coefficients for every single interval between data points, they are highly sensitive to sparse datasets and cannot reliably extrapolate maturities beyond the bounds of the observed data.

### The Nelson-Siegel Alternative
Nelson and Siegel (1987) solved this by defining the yield curve not as a series of connected dots, but as a single equation governed by four macroeconomic attributes: **Level ($\beta_0$)**, **Slope ($\beta_1$)**, **Shape ($\beta_2$)**, and **Decay ($\tau$)**.

$$r(t) = \beta_0 + \beta_1 \left( \frac{1 - e^{-t/\tau}}{t/\tau} \right) + \beta_2 \left( \frac{1 - e^{-t/\tau}}{t/\tau} - e^{-t/\tau} \right)$$

This approach is highly parsimonious. Like defining a perfect sine wave with just a few parameters, Nelson-Siegel filters out idiosyncratic noise (like a temporarily mispriced bond) and captures the true underlying shape of the term structure.

### Proving the Mechanics: The Limits
The intuition of this model is best revealed by testing its limits at extreme maturities.

**1. The Long-Term Level ($\beta_0$)**
Take the limit as time approaches infinity ($t \to \infty$). Because $t$ is in the denominator and the negative exponent of the subsequent terms, the slope and shape fractions decay to zero:
$$\lim_{t \to \infty} r(t) = \beta_0$$
This proves that $\beta_0$ serves as the macroeconomic anchor—the long-run level of interest rates.

**2. The Short-Term Slope ($\beta_1$)**
Take the limit as time approaches zero ($t \to 0$). By applying L'Hôpital's rule to the decay functions, the middle slope term goes to 1 and the final shape term goes to 0:
$$\lim_{t \to 0} r(t) = \beta_0 + \beta_1(1) + \beta_2(0) = \beta_0 + \beta_1$$
This reveals why the restricted version of the model substitutes $\beta_1$ with $(r(0) - \beta_0)$. Making that substitution guarantees the math will perfectly hit the observed overnight spot rate ($r(0)$) at time zero.

**3. The Shape ($\beta_2$) and Decay ($\tau$)**
As established above, the mathematical component tied to $\beta_2$ evaluates to zero at both $t \to 0$ and $t \to \infty$. If it begins at zero and ends at zero, its value must rise and fall in the intermediate maturities. This creates the "hump" (or curvature) in the yield curve. The parameter $\tau$ dictates exactly *where* this hump peaks and how fast the short-term slope decays.

### What about the Svensson Extension?
Lars Svensson (1994) introduced a six-parameter extension to this model, adding a second shape ($\beta_3$) and second decay ($\tau_2$) parameter to capture complex, "S-shaped" curves with double humps.

While central banks use the Svensson model to publish daily yield curves, they do so using massive datasets of highly liquid bonds. For smaller datasets, the extra flexibility introduces severe overfitting risk. The model twists and contorts to fit the pricing errors of specific bonds rather than mapping the true macroeconomic curve. Unless a clear secondary hump is visible in the raw data, the four-parameter Nelson-Siegel model remains the superior choice for stability and out-of-sample accuracy.
</details>

## 📐 Calculating The Payoff Matrix

> 💡 **Notice the Cascading Matrix**
>
> Because each row represents a synthetic bond tied to a specific maturity date, the cash flows stop exactly at the end of that contract's reference period.
>
> Once a bond hits its maturity date (the diagonal of the matrix), it returns the 100 par value along with its final coupon, and the bond terminates—resulting in zeros for all subsequent columns in that row.

To visualize this lower-triangular structure, the code below prints both the top (head) and bottom (tail) of the dataframe.

In [ ]:

# Get the total number of bonds (22: 1 overnight + 21 futures)
n_bonds = len(df_contracts)

# Extract the coupon values as a numpy array for easy indexing
coupons = df_contracts['Synthetic Coupon'].values

# Initialize a 22x22 matrix of zeros
payoff_matrix = np.zeros((n_bonds, n_bonds))

# Populate the matrix iteratively based on our cash flow logic
for i in range(n_bonds):
    if i == 0:
        # Bond 1 (Overnight SOFR): Pays par + interest on day 1 (column 0)
        payoff_matrix[i, 0] = 100 + coupons[0]
    else:
        # Bonds 2 through 22 (Futures Strip):
        # 1. They do NOT pay on the overnight date, so column 0 remains 0.
        # 2. They pay the known synthetic coupon on all intermediate quarterly IMM dates.
        for j in range(1, i):
            payoff_matrix[i, j] = coupons[j]

        # 3. They pay par (100) + the final synthetic coupon at their specific maturity.
        payoff_matrix[i, i] = 100 + coupons[i]

# Assign the populated matrix back to our existing df_payoffs dataframe
# We use the bond names as the index and the exact payment dates as the columns
df_payoffs = pd.DataFrame(payoff_matrix, index=df_contracts.index, columns=columns)

# Display the  the cascading structure for the start and end
display(df_payoffs.head())
display(df_payoffs.tail())

## 🧪 Calibrating the Nelson-Siegel Curve & Extracting Par Swap Rates

With our 22-instrument payoff matrix assembled, we now calibrate the term structure using `estimate_ns_parameters` from `financial_quant`.

The `estimate_ns_parameters` function assume continuous compounding, all internal discounting, zero rates, and boundary conditions evaluate as continuous rates ($e^{-r(t) \cdot t}$). To enforce our short-end boundary condition:

$$\lim_{t \to 0} r(t) = \beta_0 + \beta_1 = r_{\text{cont}}(0)$$

we convert the discrete overnight SOFR print into its continuously compounded equivalent using an actual calendar basis of **365.24** days:

$$r_{\text{cont}}(0) = 365.24 \cdot \ln\left(1 + \frac{r_{\text{SOFR}}}{360}\right)$$

This allows the optimizer to constrain the slope parameter directly to $\beta_1 = r_{\text{cont}}(0) - \beta_0$.

To run the calibration, we pass three primary inputs to `estimate_ns_parameters`:
1. **The Payoff Matrix:** Our $22 \times 22$ lower-triangular matrix of synthetic cash flows.
2. **Tenors in Years ($t$):** The column payment dates converted to annual year fractions measured from the anchor date using $365.24$ days per year.
3. **Observed Prices ($P$):** A vector set uniformly to $100.0$, reflecting that every synthetic bond trades at par under no-arbitrage conditions.

> ⚠️ **Rate Convention Note:** All parameters, spot rates, and theoretical forward yields returned by the optimizer will initially be in continuously compounded format. Once the continuous zero curve is established, we will extract the par swap rates and convert them back into the standard **30/360** annual/semi-annual swap convention used in fixed-income markets.

In [ ]:
# ==============================================================================
# PREPARE INPUTS AND CALIBRATE CONTINUOUS NELSON-SIEGEL MODEL
# ==============================================================================

# 1. Convert payment dates (columns of df_payoffs) to maturity in years from anchor_date
# Using an exact solar/calendar year convention of 365.24 days
payment_dates = pd.to_datetime(df_payoffs.columns)
maturities_in_years = (payment_dates - anchor_date).days / 365.24

# 2. Target prices: All 22 synthetic bonds are valued at par (100.0)
prices = pd.Series(np.full(shape=len(df_payoffs), fill_value=100.0))

# 3. Convert 1-day overnight SOFR (3.95%) directly into an annualized continuous rate
# Uses 360-day money market accrual converted onto the 365.24 continuous curve basis
sofr_simple = 0.0395
sofr_anchor = 365.24 * np.log(1.0 + sofr_simple / 360.0)
display(f'Continuous SOFR {sofr_anchor: .4f}')
display(f'First Five Maturity in Years {maturities_in_years[:5]}')
display(f'First Five Prices {prices[:5]}')

## The Estimation: Apply the constrained Nelson-Siegel model to estimate the spot rates.

With our data prepped, we define the initial parameter guesses for our optimization routine. Because we are running a restricted model that forces the very short end of our curve to equal the overnight SOFR rate, the model only requires guesses for the long-term rate ($\beta_0$), the shape/curvature ($\beta_2$), and the decay factor ($\tau$). The slope parameter ($\beta_1$) is inherently constrained by the SOFR anchor. We pass these inputs to the optimizer to minimize the Sum of Squared Residuals (SSR) and extract our final spot rates.

In [ ]:
# 1. Define guesses (Notice slope_guess is removed since it's restricted)
long_rate_guess=0.055
shape_guess =0.0
tau_guess=2.0
guesses=[long_rate_guess,shape_guess,tau_guess]


# 3. Run the estimation with the clean float
restricted = fq.estimate_ns_parameters(
    df_payoffs,
    prices,
    maturities_in_years,
    guesses,
    sofr_rate=sofr_anchor
)

# 4. Calculate restricted spot rates
restricted_spot_rates, maturity_years = fq.ns_spot_rates(
    restricted.x,
    maturities_in_years,
    sofr_rate=sofr_anchor
)

## 📈 Visualizing the Theoretical Spot Curve
With our restricted Nelson-Siegel parameters optimized, we can now generate a continuous array of theoretical spot rates across a full 5-year maturity spectrum. The code below plots these model-implied rates, allowing us to visually inspect the resulting shape of our continuous yield curve.

In [ ]:
plot_data=[restricted_spot_rates*100]
#X axis time in years
xaxis=maturities_in_years
series=['3-month SOFR Futures']
ylabel='Annual Percentage Rate'
xlabel='Maturity In Years'
size=(7,5)
markers=['']*1
colors=['b']

# use nanmin and nanmax so that nan is ignogred
y_lower=np.nanmin(plot_data)-1
y_upper=np.nanmax(plot_data)+1


ylim=[y_lower,y_upper]
title='SOFR Restricted Estimate Of Term Structure (SOFR Futures):'+anchor_date.strftime('%Y-%m-%d ')
fq.one_y_axis(xaxis, plot_data, title=title, series_labels=series, xlabel=xlabel,
              y_limits=ylim,ylabel=ylabel, markers=markers,colors=colors,
              figure_size=size)

## ⚖️ Spot Rates vs. Benchmark Par Swap Rates

With our term structure calibrated, we can extract the specific rates for standard benchmark tenors (1Y through 5Y).

To align with standard fixed-income metrics, we will present both the spot and swap rates on a **Bond Equivalent Yield (BEQ)** basis—assuming semi-annual compounding.

* For the **Swap Rate**, we simply set our payment frequency to 6 months (`freq=6`) to natively generate the BEQ yield.
* For the **Spot Rate**, we algebraically convert our model's continuous yield into a semi-annual BEQ rate: $r_{\text{BEQ}} = 2 \cdot (e^{r_{\text{cont}} / 2} - 1)$.
* Finally, we convert the discrete BEQ swap rate into the **IMM Swap Rate** using the 360-day money market convention.



---
>**Note:** To translate our theoretical Nelson-Siegel curve into tradable market quotes, we apply the `fq.calc_par_yield` function developed in Chapter Two of the *Pricing And Interest Rate Risk* Volume. This algorithm uses our continuous zero-coupon spot rates to discount future cash flows, algebraically solving for the exact fixed coupon rate that prices our synthetic swap at par.



In [ ]:
# ==============================================================================
# CALCULATE BENCHMARK SPOT VS. SWAP RATES (BEQ BASIS)
# ==============================================================================

# 1. Extract 1Y-5Y Par Swap Rates on a BEQ basis (semi-annual, freq=6 months)
swap_rates_beq = np.array([
    fq.calc_par_yield(12*i, restricted.x, sofr_rate=sofr_anchor, freq=6)
    for i in range(1, 6)
])

# 2. Extract continuous spot rates and convert to BEQ
spot_rates_cont = np.array([
    fq.ns_spot_rates(restricted.x, i, sofr_rate=sofr_anchor)[0]
    for i in range(1, 6)
])
spot_rates_beq = 2.0 * (np.exp(spot_rates_cont / 2.0) - 1.0)

# 3. Convert the BEQ swap yield to the 360-day IMM convention
imm_swap_rates_5 = swap_rates_beq * (360.0 / 365.24)

# 4. Display the final comparative table
df_benchmark = pd.DataFrame({
    'Tenor': [f'{i}Y' for i in range(1, 6)],
    'Spot Rate BEQ (%)': np.round(spot_rates_beq * 100.0, 3),
    'Swap Rate BEQ (%)': np.round(swap_rates_beq * 100.0, 3),
    'IMM Swap Rate (%)': np.round(imm_swap_rates_5 * 100.0, 3)
}).set_index('Tenor')

display(df_benchmark)

## 🏁 Conclusion: From Plumbing to Pricing

We began this notebook by observing a striking market reality: the overwhelming, multi-year liquidity of the 3-Month SOFR futures contract compared to its short-term peers.

By dissecting the institutional plumbing, we uncovered that this liquidity is not speculative—it is structural. Institutions stack these futures into multi-year strips to efficiently hedge custom OTC interest rate swaps and optimize clearinghouse capital requirements.

We replicated this institutional process step-by-step:
1. **Stacking the Strip:** We linked consecutive quarterly contracts to lock in forward borrowing costs, transforming variable floating-rate exposure into known synthetic bond cash flows.
2. **Building the Payoff Matrix:** We mapped these cascading cash flows into a strict, lower-triangular matrix architecture.
3. **Bootstrapping the Curve:** We applied the restricted Nelson-Siegel model to our matrix, enforcing a continuous overnight anchor to extract the underlying theoretical spot curve.
4. **Extracting the Market:** Finally, we used that continuous spot curve to price the discrete, standard 1Y–5Y par swap rates that institutions trade every day.

While the 3-Month contract dictates the multi-year term structure, the absolute front end of the yield curve is governed by 1-Month SOFR and 30-Day Fed Funds. In the next notebook, we will explore how their unique arithmetic averaging structures are used to navigate the push and pull of short-term financial plumbing.